# Python 上下文管理器详解（with 语句）—— 教学级 Notebook

> **一句话本质**：上下文管理器 = 「进入时准备（`__enter__`），离开时清理（`__exit__`）」，无论正常结束还是发生异常，**清理动作必然执行**。

## 学习目标

学完本 Notebook，你应该能够：

1. 默写上下文管理器协议（`__enter__` / `__exit__` 的参数与返回值语义）；
2. 用**类**和 `@contextmanager` 两种方式实现生产可用的上下文管理器，并知道何时选哪种；
3. 熟练使用 `contextlib` 标准库六大工具：`suppress` / `closing` / `redirect_stdout` / `ExitStack` / `nullcontext` / `ContextDecorator`；
4. 区分同步（`with`）与异步（`async with`）上下文管理器，不混用；
5. 在 LLMOps 工程场景中（API 会话、流式响应、批量任务、模型切换、计时计费）组合运用上述知识；
6. 识别并规避 6 个高频陷阱。

> 📌 全程示例围绕 **LLMOps 业务场景**（LLM API 调用、流式输出、批量推理、模型灰度降级、用量统计）取材，所有代码单元均已真实执行，输出已嵌入。

## 1. 前置知识

> **本章解决什么问题**：上下文管理器依赖四个语言基础——生成器、`try/finally`、装饰器、`sys.exc_info()`。先把这四块地基打牢，后面所有内容都是它们的组合。

| 前置概念 | 在上下文管理器中的作用 |
| --- | --- |
| 生成器（`yield`） | `@contextmanager` 的底座：`yield` 前是进入逻辑，后是退出逻辑 |
| `try/finally` | `with` 语句的底层语义就是 `try/finally`，保证清理必然执行 |
| 装饰器（`@`） | `@contextmanager` 本身是一个装饰器，把生成器函数包装成上下文管理器工厂 |
| `sys.exc_info()` | `__exit__` 收到的 `(exc_type, exc_val, exc_tb)` 三参数正是来自它 |

### 1.1 生成器：`yield` 的暂停与恢复

`@contextmanager` 要求生成器**恰好 yield 一次**：`yield` 把函数切成「进入 / 退出」两半。先用最小例子验证 yield 的执行时机。

In [1]:
# 知识点 1-1：生成器函数调用时不执行函数体，next() 驱动执行，yield 处暂停
def demo():
    print("  [函数体] 第一段：第一次 next 才执行")
    yield 1                        # 第一个暂停点
    print("  [函数体] 第二段：第二次 next 才执行")
    yield 2                        # 第二个暂停点
    print("  [函数体] 收尾：函数即将结束")

print("① 调用 demo()，只得到生成器对象，函数体一行都没跑")
gen = demo()
print("② next ->", next(gen))      # 驱动到第一个 yield
print("③ next ->", next(gen))      # 驱动到第二个 yield
try:
    next(gen)                      # 函数结束
except StopIteration:
    print("④ 函数执行完毕，抛出 StopIteration")

① 调用 demo()，只得到生成器对象，函数体一行都没跑
  [函数体] 第一段：第一次 next 才执行
② next -> 1
  [函数体] 第二段：第二次 next 才执行
③ next -> 2
  [函数体] 收尾：函数即将结束
④ 函数执行完毕，抛出 StopIteration


### 1.2 `try/finally`：清理必然执行

`with` 能「无论异常与否都清理」，本质就是编译器把 `with` 块包进了 `try/finally`。验证 finally 的两个关键事实：**正常 return 时执行、异常抛出时也执行**。

In [2]:
# 知识点 1-2：finally 在「正常返回」和「异常抛出」两条路径上都会执行
def work(should_fail):
    try:
        if should_fail:
            raise ValueError("模拟业务失败")
        return "正常返回值"
    finally:
        print(f"  [finally] should_fail={should_fail} -> 我依然执行了")

print(work(False))          # 注意：finally 的打印先于返回值被外层接收
try:
    work(True)
except ValueError as e:
    print("外层捕获:", e)

  [finally] should_fail=False -> 我依然执行了
正常返回值
  [finally] should_fail=True -> 我依然执行了
外层捕获: 模拟业务失败


### 1.3 装饰器：`@` 语法

`@contextmanager` 就是一个装饰器：它接收一个生成器函数，返回一个「调用后得到上下文管理器」的新函数。先回顾装饰器的包装结构。

In [3]:
# 知识点 1-3：装饰器 = 接收函数、返回包装函数
import functools

def log_call(fn):
    @functools.wraps(fn)            # 保留原函数的 __name__ 等元信息
    def wrapper(*args, **kwargs):
        print(f"-> 进入 {fn.__name__}")
        result = fn(*args, **kwargs)
        print(f"<- 离开 {fn.__name__}，返回 {result}")
        return result
    return wrapper

@log_call
def add(a, b):
    return a + b

add(2, 3)
# 对照记忆：@contextmanager 做的事结构相同——进入时跑 yield 前逻辑，离开时跑 yield 后逻辑

-> 进入 add
<- 离开 add，返回 5


5

### 1.4 `sys.exc_info()`：异常三要素

`__exit__(self, exc_type, exc_val, exc_tb)` 的三个参数不是魔法，就是 `sys.exc_info()` 的返回值。

In [4]:
# 知识点 1-4：sys.exc_info() 返回 (异常类型, 异常实例, traceback)
import sys

try:
    1 / 0
except ZeroDivisionError:
    exc_type, exc_val, exc_tb = sys.exc_info()
    print("异常类型  :", exc_type.__name__)
    print("异常实例  :", exc_val)
    print("traceback :", type(exc_tb).__name__, "| 出错行号:", exc_tb.tb_lineno)

# 没有异常时三者都是 None —— 这正是 __exit__ 判断"是否发生异常"的依据

异常类型  : ZeroDivisionError
异常实例  : division by zero
traceback : traceback | 出错行号: 5


## 2. 上下文管理器协议

> **本章解决什么问题**：`with` 语句到底调用了什么？掌握协议，就掌握了上下文管理器的一切。

任何对象只要实现下面两个方法，就是上下文管理器：

```python
class ContextManager:
    def __enter__(self):
        # 进入 with 时调用；返回值赋给 as 后面的变量
        return resource

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 离开 with 时调用（无论是否异常）
        # 返回 True  -> 异常被抑制，不再向外传播
        # 返回 False/None -> 异常继续传播
        return False
```

**五个关键语义**：

| # | 规则 |
| --- | --- |
| 1 | `__enter__` 在进入 `with` 块**之前**调用一次 |
| 2 | `__exit__` 在离开 `with` 块时调用**且仅调用一次**，即使发生异常 |
| 3 | `as 变量` 拿到的是 `__enter__` 的**返回值**，不一定是管理器本身 |
| 4 | 无异常时 `__exit__` 三参数全为 `None`；有异常时依次为异常类型 / 实例 / traceback |
| 5 | `__exit__` 返回真值会**吞掉异常**，慎用 |

In [5]:
# 知识点 2-1：用打印轨迹验证协议的调用顺序与参数
class TraceCM:
    """跟踪型上下文管理器：把协议方法的每次调用打印出来"""
    def __enter__(self):
        print("[__enter__] 进入 with，准备资源")
        return "资源对象"              # 这个值会赋给 as 变量
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"[__exit__] exc_type={exc_type}, exc_val={exc_val}, exc_tb={exc_tb}")
        return False                   # 不抑制任何异常

print("--- 场景 A：无异常 ---")
with TraceCM() as res:
    print(f"[with 块内] as 拿到: {res!r}")
print("[with 块外] 已离开")

print("\n--- 场景 B：有异常，返回 False 继续传播 ---")
try:
    with TraceCM():
        raise KeyError("缺少 api_key")
except KeyError as e:
    print("[外层捕获]", e)

--- 场景 A：无异常 ---
[__enter__] 进入 with，准备资源
[with 块内] as 拿到: '资源对象'
[__exit__] exc_type=None, exc_val=None, exc_tb=None
[with 块外] 已离开

--- 场景 B：有异常，返回 False 继续传播 ---
[__enter__] 进入 with，准备资源
[__exit__] exc_type=<class 'KeyError'>, exc_val='缺少 api_key', exc_tb=<traceback object at 0x1096e50c0>
[外层捕获] '缺少 api_key'


In [6]:
# 知识点 2-2：验证 __exit__ 返回 True 会抑制异常（返回 False 则传播）
class InspectCM:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            print(f"  异常类型={exc_type.__name__} 实例={exc_val} 出错行={exc_tb.tb_lineno}")
        return False                    # 不吞异常

class SwallowCM:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"  收到异常 {exc_type.__name__}，返回 True 将其吞掉")
        return True                     # 吞掉异常

print("--- InspectCM（返回 False）---")
try:
    with InspectCM():
        raise KeyError("缺少 api_key")
except KeyError as e:
    print("  异常传播到外层:", e)

print("--- SwallowCM（返回 True）---")
with SwallowCM():
    raise RuntimeError("这个异常不会出现")
print("  with 块之后代码继续执行 —— 异常已被抑制")

--- InspectCM（返回 False）---
  异常类型=KeyError 实例='缺少 api_key' 出错行=20
  异常传播到外层: '缺少 api_key'
--- SwallowCM（返回 True）---
  收到异常 RuntimeError，返回 True 将其吞掉
  with 块之后代码继续执行 —— 异常已被抑制


### ⚠️ 修正说明：源文档中 `with` 的等价展开式并不准确

源文档给出的等价代码是：

```python
# ❌ 源文档写法（不准确）
try:
    do_something(value)
except Exception as e:
    if not mgr.__exit__(type(e), e, e.__traceback__):
        raise
else:
    mgr.__exit__(None, None, None)
```

问题有两个：

1. **只捕获 `Exception`**：若 `with` 块内抛出 `KeyboardInterrupt`、`SystemExit` 等 `BaseException`，`__exit__` 将**完全不会被调用**——这与真实语义不符；
2. 真实语义基于 `try/finally`：无论何种退出路径（正常 return、break、任何异常），`__exit__` 必然执行。

**PEP 343 定义的准确语义**（简化）：

```python
mgr = EXPR
value = type(mgr).__enter__(mgr)
exc = True
try:
    BLOCK(value)          # with 块
except:
    exc = False
    if not type(mgr).__exit__(mgr, *sys.exc_info()):
        raise
finally:
    if exc:
        type(mgr).__exit__(mgr, None, None, None)
```

下面用 `KeyboardInterrupt`（继承自 `BaseException`，不属于 `Exception`）做决定性验证。

In [7]:
# 知识点 2-3（修正验证）：BaseException 也会触发 __exit__
class RecordCM:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        name = exc_type.__name__ if exc_type else None
        print(f"  [__exit__] 被调用，exc_type={name}")
        return False

# KeyboardInterrupt 不是 Exception 的子类
print("Exception 是 KeyboardInterrupt 的父类吗:", issubclass(KeyboardInterrupt, Exception))
try:
    with RecordCM():
        raise KeyboardInterrupt          # 模拟用户按 Ctrl+C
except KeyboardInterrupt:
    print("  [外层捕获] KeyboardInterrupt —— 证明 __exit__ 依然被执行了")
# 若按源文档的 except Exception 展开，这里 __exit__ 不会执行，资源就泄漏了

Exception 是 KeyboardInterrupt 的父类吗: False
  [__exit__] 被调用，exc_type=KeyboardInterrupt
  [外层捕获] KeyboardInterrupt —— 证明 __exit__ 依然被执行了


## 3. 用类实现上下文管理器

> **本章解决什么问题**：需要保存状态、逻辑复杂、要被多处复用时，用类实现是最标准的方式。

三个经典案例：**文件管理器**（资源释放）、**计时器**（进入/离开记录状态）、**异常抑制器**（`__exit__` 返回值的运用）。

In [8]:
# 知识点 3-1：文件管理器 —— 保证文件必然关闭
import tempfile, os

class ManagedFile:
    def __init__(self, path, mode="r", encoding="utf-8"):
        self.path = path          # 构造时只记录参数，不打开文件
        self.mode = mode
        self.encoding = encoding
        self.file = None
    def __enter__(self):
        self.file = open(self.path, self.mode, encoding=self.encoding)
        return self.file          # as 变量拿到的是文件对象，不是 self
    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()     # 无论是否异常都关闭
        return False

tmpdir = tempfile.mkdtemp()
path = os.path.join(tmpdir, "prompt.txt")

with ManagedFile(path, "w") as f:
    f.write("请总结这篇文档")
print("写完后文件已关闭:", f.closed)

with ManagedFile(path) as f2:
    print("读回内容:", f2.read())
print("读完后文件已关闭:", f2.closed)

os.remove(path); os.rmdir(tmpdir)

写完后文件已关闭: True
读回内容: 请总结这篇文档
读完后文件已关闭: True


In [9]:
# 知识点 3-2：计时器 —— __enter__ 返回 self，离开后可读取统计结果
import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self               # 返回 self，as 变量拿到管理器本身
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"[Timer] 块内耗时: {self.elapsed:.4f} 秒")
        return False

with Timer() as t:                # t 就是 Timer 实例
    total = sum(i * i for i in range(300_000))

print(f"[Timer] with 块外仍可读取: {t.elapsed:.4f} 秒")
# 注：计时为实测值，每次运行略有浮动属正常现象

[Timer] 块内耗时: 0.0084 秒
[Timer] with 块外仍可读取: 0.0084 秒


In [10]:
# 知识点 3-3：异常抑制器 —— 精确控制 __exit__ 的返回值
class Suppress:
    """只抑制指定的异常类型，其余异常照常传播"""
    def __init__(self, *exceptions):
        self.exceptions = exceptions
    def __enter__(self):
        return None
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None and issubclass(exc_type, self.exceptions):
            print(f"  [Suppress] 已抑制 {exc_type.__name__}")
            return True           # 命中 -> 抑制
        return False              # 未命中 -> 传播

with Suppress(ValueError):
    int("abc")                    # ValueError 被抑制
print("① 继续执行")

try:
    with Suppress(ValueError):
        raise KeyError("非预期异常")
except KeyError as e:
    print("② KeyError 不在抑制名单，正常传播:", e)

  [Suppress] 已抑制 ValueError
① 继续执行
② KeyError 不在抑制名单，正常传播: '非预期异常'


## 4. 用 `@contextmanager` 实现（生成器方式）

> **本章解决什么问题**：简单的「准备/清理」逻辑没必要写一个类，用生成器函数 5 行搞定。

```python
from contextlib import contextmanager

@contextmanager
def managed_resource():
    resource = acquire()   # ① yield 之前 = __enter__ 逻辑
    try:
        yield resource     # ② yield 的值 -> as 变量；必须恰好 yield 一次
    finally:
        release(resource)  # ③ yield 之后 = __exit__ 逻辑，放 finally 保证执行
```

**四条硬性规则**：

| # | 规则 | 违反后果 |
| --- | --- | --- |
| 1 | 生成器必须**恰好 yield 一次** | `RuntimeError: generator didn't stop / didn't yield` |
| 2 | 清理逻辑放进 `finally` | 异常时清理被跳过，资源泄漏 |
| 3 | `with` 块内的异常会**在 yield 处重新抛出** | 可在生成器内 `try/except` 捕获处理 |
| 4 | 想抑制异常：在生成器内捕获后**不重新抛出** | 与类版返回 True 等价，但更隐蔽，需注释说明 |

In [11]:
# 知识点 4-1：生成器版文件管理器 + 验证异常在 yield 处抛出
from contextlib import contextmanager
import tempfile, os

@contextmanager
def managed_file(path, mode="r", encoding="utf-8"):
    f = open(path, mode, encoding=encoding)   # 进入逻辑
    try:
        yield f                                # as 变量拿到 f
    finally:
        f.close()                              # 退出逻辑（finally 保证执行）

tmpdir = tempfile.mkdtemp()
path = os.path.join(tmpdir, "note.txt")
with managed_file(path, "w") as f:
    f.write("hello llmops")
print("① 退出后文件 closed =", f.closed)

# 验证：with 块内的异常会在 yield 处抛出，生成器可以捕获
@contextmanager
def observe():
    try:
        yield
    except ValueError as e:
        print(f"② 生成器在 yield 处收到异常: {e}")
        raise                                  # 处理完重新抛出（不抑制）

try:
    with observe():
        raise ValueError("来自 with 块的异常")
except ValueError:
    print("③ 异常被重新抛到外层")

os.remove(path); os.rmdir(tmpdir)

① 退出后文件 closed = True
② 生成器在 yield 处收到异常: 来自 with 块的异常
③ 异常被重新抛到外层


In [12]:
# 知识点 4-2：事务管理器 —— LLM 调用日志落库时的「成功提交 / 失败回滚」
import sqlite3
from contextlib import contextmanager

@contextmanager
def transaction(conn):
    try:
        yield conn
    except Exception:
        conn.rollback()
        print("  [事务] 发生异常 -> 已回滚")
        raise                      # 回滚后仍要向外抛，让调用方知道失败了
    else:
        conn.commit()
        print("  [事务] 无异常 -> 已提交")

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE prompt_log (id INTEGER, content TEXT)")

with transaction(conn):
    conn.execute("INSERT INTO prompt_log VALUES (1, '正常写入的 prompt')")

try:
    with transaction(conn):
        conn.execute("INSERT INTO prompt_log VALUES (2, '这条会回滚')")
        raise RuntimeError("LLM 接口超时，写入作废")
except RuntimeError as e:
    print("  [外层]", e)

print("最终表内数据:", conn.execute("SELECT * FROM prompt_log").fetchall())
conn.close()

  [事务] 无异常 -> 已提交
  [事务] 发生异常 -> 已回滚
  [外层] LLM 接口超时，写入作废
最终表内数据: [(1, '正常写入的 prompt')]


## 5. 多个上下文管理器：进入从左到右，退出从右到左

> **本章解决什么问题**：一行 `with` 管理多个资源时，顺序如何保证？

```python
with open("a.txt") as f1, open("b.txt") as f2:   # 等价于嵌套两个 with
    ...
```

顺序规则：**进入从左到右（A→B→C），退出从右到左（C→B→A），后进先出（LIFO，栈语义）**。这也是 `ExitStack`（第 6.4 节）名字的由来。下面用代码验证。

In [13]:
# 知识点 5-1：验证多个上下文管理器的进入/退出顺序
class Step:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print(f"进入 {self.name}")
        return self
    def __exit__(self, *exc):
        print(f"退出 {self.name}")
        return False

with Step("A"), Step("B"), Step("C"):
    print("  工作中（A/B/C 都已就绪）")
# 预期输出：进入 A B C -> 工作 -> 退出 C B A

进入 A
进入 B
进入 C
  工作中（A/B/C 都已就绪）
退出 C
退出 B
退出 A


## 6. 标准库 `contextlib` 六大工具

> **本章解决什么问题**：80% 的上下文管理器需求，标准库已经写好了，不要重复造轮子。

| 工具 | 一句话用途 | LLMOps 典型场景 |
| --- | --- | --- |
| `suppress(*exc)` | 抑制指定异常 | 忽略"缓存文件不存在"、忽略可重试的限流 |
| `closing(obj)` | 给只有 `close()` 的对象补上协议 | 第三方 HTTP/DB 响应对象 |
| `redirect_stdout(f)` | 临时重定向标准输出 | 捕获 LLM 框架打印的日志做归集 |
| `ExitStack()` | 动态管理**数量不定**的资源 | 批量打开 N 个 prompt 文件 |
| `nullcontext()` | 什么都不做的占位 CM | 让"可选加锁/可选追踪"的代码统一写法 |
| `ContextDecorator` | 让 CM 同时能当装饰器 | 给 LLM 调用函数一键挂计时 |

In [14]:
# 知识点 6-1：suppress —— 用一行替代啰嗦的 try/except/pass
from contextlib import suppress
import os

# 场景：清理临时缓存文件，文件不存在也无所谓
with suppress(FileNotFoundError):
    os.remove("definitely_not_exist_12345.cache")
print("① 文件不存在也不报错，继续执行")

# 等价写法对比（不用 suppress 时）：
# try:
#     os.remove(...)
# except FileNotFoundError:
#     pass

# suppress 只抑制名单内的异常
try:
    with suppress(FileNotFoundError):
        raise PermissionError("权限不足")
except PermissionError as e:
    print("② 其他异常正常传播:", e)

① 文件不存在也不报错，继续执行
② 其他异常正常传播: 权限不足


In [15]:
# 知识点 6-2：closing —— 给"有 close() 但不是上下文管理器"的对象补协议
from contextlib import closing
import sqlite3

# sqlite3.Connection 作为 CM 只管理事务（见 14 章陷阱），不会关闭连接；
# closing 保证 with 结束时调用 .close()
with closing(sqlite3.connect(":memory:")) as conn:
    conn.execute("CREATE TABLE t (x INTEGER)")
    conn.execute("INSERT INTO t VALUES (42)")
    print("查询结果:", conn.execute("SELECT x FROM t").fetchone())
print("with 结束后连接已关闭")

查询结果: (42,)
with 结束后连接已关闭


In [16]:
# 知识点 6-3：redirect_stdout —— 捕获第三方库打印的日志
from contextlib import redirect_stdout
import io

buf = io.StringIO()
with redirect_stdout(buf):
    print("[llm-sdk] connecting...")
    print("[llm-sdk] stream finished")   # 这些 print 被重定向进 buf

captured = buf.getvalue()
print("捕获到的日志行数:", len(captured.strip().splitlines()))
print("第一行:", captured.strip().splitlines()[0])
# 姊妹工具：redirect_stderr 重定向标准错误

捕获到的日志行数: 2
第一行: [llm-sdk] connecting...


In [17]:
# 知识点 6-4：ExitStack —— 资源数量不确定时的唯一干净方案
from contextlib import ExitStack
import tempfile, os

tmpdir = tempfile.mkdtemp()
paths = []
for i in range(3):
    p = os.path.join(tmpdir, f"prompt_{i}.txt")
    with open(p, "w", encoding="utf-8") as f:
        f.write(f"prompt 内容 {i}")
    paths.append(p)

# 场景：批量加载 N 个 prompt 文件（N 运行时才确定）
with ExitStack() as stack:
    files = [stack.enter_context(open(p, encoding="utf-8")) for p in paths]
    print(f"① 动态打开了 {len(files)} 个文件")
    print("② 首个文件内容:", files[0].read())
print("③ 退出后全部自动关闭:", all(f.closed for f in files))

# callback：注册任意清理函数，同样后进先出
with ExitStack() as stack:
    stack.callback(print, "清理 1")
    stack.callback(print, "清理 2")
    print("④ 工作中")

for p in paths:
    os.remove(p)
os.rmdir(tmpdir)

① 动态打开了 3 个文件
② 首个文件内容: prompt 内容 0
③ 退出后全部自动关闭: True
④ 工作中
清理 2
清理 1


In [18]:
# 知识点 6-5：nullcontext —— 让"可选"逻辑写法统一
from contextlib import nullcontext
import threading

lock = threading.Lock()

def process(data, use_lock):
    # use_lock 为 False 时用 nullcontext 占位，with 语句无需分支
    cm = lock if use_lock else nullcontext()
    with cm:
        print(f"use_lock={use_lock}: 处理 {data}")

process("批量推理任务", use_lock=True)
process("本地调试任务", use_lock=False)

# 进阶：nullcontext(enter_result) 可指定 as 变量的值
with nullcontext("默认回复") as reply:
    print("as 变量:", reply)

use_lock=True: 处理 批量推理任务
use_lock=False: 处理 本地调试任务
as 变量: 默认回复


## 7. 异步上下文管理器（`async with`）

> **本章解决什么问题**：LLM SDK 的异步客户端（`aiohttp`、`openai.AsyncOpenAI` 等）怎么管理？用 `__aenter__` / `__aexit__`。

| 对比项 | 同步 | 异步 |
| --- | --- | --- |
| 协议方法 | `__enter__` / `__exit__` | `__aenter__` / `__aexit__` |
| 语句 | `with` | `async with` |
| 快捷写法 | `@contextmanager` | `@asynccontextmanager` |
| 能否混用 | ❌ 同步 CM 不能用于 `async with` | ❌ 异步 CM 不能用于 `with` |

> 注：Jupyter 的 code cell 支持顶层 `await`；在 `.py` 脚本中请改用 `asyncio.run(main())`。

In [19]:
# 知识点 7-1：类实现异步上下文管理器 —— 模拟异步 LLM 连接
import asyncio

class AsyncLLMConn:
    async def __aenter__(self):
        print("异步建立连接 (模拟)")
        await asyncio.sleep(0.01)        # 模拟建连耗时
        return self
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("异步释放连接")
        return False
    async def chat(self, prompt):
        await asyncio.sleep(0.01)        # 模拟网络往返
        return f"[回复] {prompt}"

async def main():
    async with AsyncLLMConn() as conn:
        reply = await conn.chat("什么是上下文管理器")
        print("拿到", reply)

await main()          # Jupyter 顶层 await；脚本中写 asyncio.run(main())

异步建立连接 (模拟)
拿到 [回复] 什么是上下文管理器
异步释放连接


In [20]:
# 知识点 7-2：@asynccontextmanager —— 生成器方式的异步版
import asyncio
import time
from contextlib import asynccontextmanager

@asynccontextmanager
async def async_timer(label):
    start = time.perf_counter()
    print("进入")
    try:
        yield 42                          # as 变量拿到 42
    finally:
        print(f"退出，{label} 耗时 {time.perf_counter() - start:.4f} 秒")

async def main2():
    async with async_timer("异步推理") as value:
        await asyncio.sleep(0.05)
        print("as 变量 =", value)

await main2()

进入


as 变量 = 42
退出，异步推理 耗时 0.0511 秒


## 8. 工程高频场景速览

> **本章解决什么问题**：把协议和标准库落到日常代码里。以下 5 个场景覆盖了后端/LLMOps 开发中 90% 的 `with` 用法。

1. **线程锁** —— `threading.Lock` 的 `__enter__` 获取锁、`__exit__` 释放锁；
2. **临时目录** —— `tempfile.TemporaryDirectory()` 退出即删除，压测/缓存首选；
3. **临时切换工作目录** —— 自定义 `chdir`，用 `finally` 保证恢复；
4. **临时修改环境变量** —— LLM 应用切换模型/密钥的常用手段；
5. **测试 mock** —— `unittest.mock.patch` 本身就是上下文管理器。

In [21]:
# 知识点 8-1：线程锁 —— with 保证锁必然释放（确定性演示）
import threading

counter = 0
lock = threading.Lock()

def worker():
    global counter
    for _ in range(10000):
        with lock:                 # __enter__ 获取锁
            counter += 1           # 临界区
        # __exit__ 释放锁；若此处抛异常锁也会释放

threads = [threading.Thread(target=worker) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print("4 线程各累加 10000 次，加锁结果:", counter)   # 恒为 40000

4 线程各累加 10000 次，加锁结果: 40000


In [22]:
# 知识点 8-2：TemporaryDirectory —— 临时缓存目录，退出自动删除
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    cache_path = os.path.join(tmpdir, "embedding_cache.json")
    with open(cache_path, "w") as f:
        f.write('{"cached": true}')
    print("块内目录内容:", os.listdir(tmpdir))

print("退出后目录仍存在吗:", os.path.exists(tmpdir))   # False：已自动清理

块内目录内容: ['embedding_cache.json']
退出后目录仍存在吗: False


In [23]:
# 知识点 8-3：临时切换工作目录 —— finally 保证必然恢复
import os
from contextlib import contextmanager

@contextmanager
def chdir(path):
    old = os.getcwd()      # 进入：保存现场
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)      # 退出：恢复现场

print("原目录  :", os.getcwd())
with chdir("/tmp"):
    print("临时切换:", os.getcwd())
print("恢复为  :", os.getcwd())

原目录  : /Users/huangyoubin/Documents/LLMOps/llmops-api/study/0-python基础
临时切换: /private/tmp
恢复为  : /Users/huangyoubin/Documents/LLMOps/llmops-api/study/0-python基础


In [24]:
# 知识点 8-4：临时修改环境变量 —— LLM 应用切换模型的核心手段
import os
from contextlib import contextmanager

@contextmanager
def env(**kwargs):
    old = {}
    for k, v in kwargs.items():
        old[k] = os.environ.get(k)   # 记录旧值（可能是 None）
        os.environ[k] = str(v)
    try:
        yield
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k, None)   # 原本不存在 -> 删除
            else:
                os.environ[k] = v         # 原本存在 -> 还原

with env(LLM_MODEL="qwen-plus", LLM_BASE_URL="https://api.example.com"):
    print("块内:", os.environ["LLM_MODEL"], "|", os.environ["LLM_BASE_URL"])
print("块外 LLM_MODEL =", os.environ.get("LLM_MODEL"), "（已自动清理）")

块内: qwen-plus | https://api.example.com
块外 LLM_MODEL = None （已自动清理）


In [25]:
# 知识点 8-5：patch —— 测试时替换 LLM 调用，退出自动恢复
from unittest.mock import patch

class LLMClient:
    def chat(self, prompt):
        return f"真实回复(要花钱): {prompt}"

client = LLMClient()

# patch 本身是上下文管理器，退出 with 自动还原被替换的对象
with patch.object(LLMClient, "chat", return_value="mock 回复（免费且确定）"):
    print("测试中:", client.chat("讲个笑话"))

print("测试外:", client.chat("讲个笑话"))

测试中: mock 回复（免费且确定）
测试外: 真实回复(要花钱): 讲个笑话


## 9. `ContextDecorator`：一个类，既是上下文管理器又是装饰器

> **本章解决什么问题**：想让「计时/日志」能力既能 `with` 块用、又能 `@` 挂到函数上？继承 `ContextDecorator` 即可，无需改动 `__enter__`/`__exit__` 的任何逻辑。

| 用法 | 写法 |
| --- | --- |
| 上下文管理器 | `with Timer(): ...` |
| 装饰器 | `@Timer()` <br> `def work(): ...` |

⚠️ 注意：作为装饰器时**每次调用函数都会新建实例**，因此不能在 `__enter__` 里指望与 `with` 用法共享实例状态。

In [26]:
# 知识点 9-1：Timer 两种用法同源
from contextlib import ContextDecorator
import time

class Timer(ContextDecorator):
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"耗时: {time.perf_counter() - self.start:.4f} 秒")
        return False

# 用法一：装饰器
@Timer()
def embed_texts():
    sum(range(500_000))
    return "embedding 完成"

print(embed_texts())

# 用法二：with 块
with Timer():
    sum(range(500_000))
print("rerank 完成")

耗时: 0.0038 秒
embedding 完成
耗时: 0.0037 秒
rerank 完成


## 10. 反例单元：错误写法 vs 正确写法

> **本章解决什么问题**：每个易错点先看「错误写法」的实际症状（真实触发的报错/静默错误已捕获打印，保证本 Notebook 零报错），再看「正确写法」对比差异。

| 反例 | 症状 |
| --- | --- |
| 10.1 `__exit__` 无脑返回 True | bug 被静默吞掉 |
| 10.2 误以为 `as` 变量是管理器本身 | 拿到意外类型 |
| 10.3 生成器 yield 次数不对 | `RuntimeError` |
| 10.4 `__exit__` 里再抛异常 | 覆盖原始异常 |
| 10.5 同步/异步 CM 混用 | `TypeError` |

### 10.1 ❌ `__exit__` 无条件返回 True

In [27]:
# ❌ 错误写法：所有异常被静默吞掉
class DangerousCM:
    def __enter__(self):
        return self
    def __exit__(self, *exc):
        return True          # 吞掉一切异常，包括你没想到的

with DangerousCM():
    result = 1 / 0           # 这里出错了，但没有任何报错！
print("❌ 症状：除零错误被静默吞掉，程序'正常'继续，bug 无从排查")

# ✅ 正确写法：只抑制明确预期的异常类型
from contextlib import suppress

with suppress(ZeroDivisionError):
    1 / 0
print("✅ 明确抑制 ZeroDivisionError")

try:
    with suppress(ZeroDivisionError):
        raise KeyError("非预期异常")
except KeyError as e:
    print("✅ 非预期异常 KeyError 正常传播:", e)

❌ 症状：除零错误被静默吞掉，程序'正常'继续，bug 无从排查
✅ 明确抑制 ZeroDivisionError
✅ 非预期异常 KeyError 正常传播: '非预期异常'


### 10.2 ❌ 误以为 `as` 变量就是管理器本身

In [28]:
# ❌ 错误认知：with X() as y 中 y 一定是 X 的实例
class Demo:
    def __enter__(self):
        return 123           # 返回值可以是任意对象
    def __exit__(self, *exc):
        return False

with Demo() as x:
    print("as 变量 =", x, "| 类型:", type(x).__name__)   # 123, int —— 不是 Demo！

# ✅ 正确认知：as 变量 = __enter__ 的返回值；需要管理器本身时显式 return self
class DemoSelf:
    def __enter__(self):
        return self
    def __exit__(self, *exc):
        return False

with DemoSelf() as y:
    print("返回 self 时:", type(y).__name__)             # DemoSelf

as 变量 = 123 | 类型: int
返回 self 时: DemoSelf


### 10.3 ❌ 生成器没有恰好 yield 一次

In [29]:
# ❌ 错误写法 1：yield 了两次
from contextlib import contextmanager

@contextmanager
def bad_two_yields():
    yield 1
    yield 2

try:
    with bad_two_yields():
        pass
except RuntimeError as e:
    print("❌ yield 两次 ->", type(e).__name__, ":", e)

# ❌ 错误写法 2：是生成器，但 yield 前直接 return（永远执行不到 yield）
# 注：若函数体里完全没有 yield，它根本不是生成器函数，会报 TypeError 而非 RuntimeError
@contextmanager
def bad_no_yield():
    if True:
        return          # 提前 return
    yield               # 语法上存在，保证它是生成器函数

try:
    with bad_no_yield():
        pass
except RuntimeError as e:
    print("❌ 没有 yield ->", type(e).__name__, ":", e)

# ✅ 正确写法：恰好 yield 一次
@contextmanager
def good_one_yield():
    yield "恰好一次"

with good_one_yield() as v:
    print("✅ 正确:", v)

❌ yield 两次 -> RuntimeError : generator didn't stop
❌ 没有 yield -> RuntimeError : generator didn't yield
✅ 正确: 恰好一次


### 10.4 ❌ `__exit__` 中释放资源时又抛异常，覆盖原异常

In [30]:
# ❌ 错误写法：清理逻辑抛异常，业务代码的原始异常被顶掉
class FragileCM:
    def __enter__(self):
        return self
    def __exit__(self, *exc):
        raise OSError("释放资源失败")      # 新的异常覆盖原异常

try:
    with FragileCM():
        raise ValueError("业务代码的原始异常")
except Exception as e:
    print("❌ 外层看到的异常:", type(e).__name__, "-", e)
    print("   原始异常只能从 __context__ 里找:", type(e.__context__).__name__, "-", e.__context__)

# ✅ 正确写法：清理逻辑自身 try/except 保护，失败只记日志
class RobustCM:
    def __enter__(self):
        return self
    def __exit__(self, *exc):
        try:
            raise OSError("释放资源失败")  # 模拟清理失败
        except OSError as cleanup_err:
            print("   [清理失败，已记录日志]", cleanup_err)
        return False                        # 不干扰原异常传播

try:
    with RobustCM():
        raise ValueError("业务代码的原始异常")
except ValueError as e:
    print("✅ 外层看到的异常:", type(e).__name__, "-", e)

❌ 外层看到的异常: OSError - 释放资源失败
   原始异常只能从 __context__ 里找: ValueError - 业务代码的原始异常
   [清理失败，已记录日志] 释放资源失败
✅ 外层看到的异常: ValueError - 业务代码的原始异常


### 10.5 ❌ 同步 / 异步上下文管理器混用

In [31]:
# ❌ 错误写法 1：同步 CM 用于 async with
class SyncCM:
    def __enter__(self): return self
    def __exit__(self, *exc): return False

async def mix1():
    try:
        async with SyncCM():
            pass
    except TypeError as e:
        print("❌ 同步 CM + async with ->", type(e).__name__, ":", str(e)[:60], "...")

await mix1()

# ❌ 错误写法 2：异步 CM 用于 with
class AsyncCM:
    async def __aenter__(self): return self
    async def __aexit__(self, *exc): return False

try:
    with AsyncCM():
        pass
except TypeError as e:
    print("❌ 异步 CM + with ->", type(e).__name__, ":", str(e)[:60], "...")

# ✅ 正确写法：协议与语句一一对应
print("✅ 规则：__enter__/__exit__ 配 with；__aenter__/__aexit__ 配 async with")

❌ 同步 CM + async with -> TypeError : 'SyncCM' object does not support the asynchronous context ma ...
❌ 异步 CM + with -> TypeError : 'AsyncCM' object does not support the context manager protoc ...
✅ 规则：__enter__/__exit__ 配 with；__aenter__/__aexit__ 配 async with


## 11. 实用示例集：LLMOps 五大高频模式

> **本章解决什么问题**：把前面的零件组装成可以直接抄进工程的 5 个模式。

| # | 模式 | 适用场景 | 注意点 |
| --- | --- | --- | --- |
| 1 | LLM 客户端会话管理 | 需要统计调用量/计费、确保连接释放 | `__enter__` 返回 self，退出时输出统计 |
| 2 | 流式响应读取 | 第三方响应对象只有 `close()` | 用 `closing` 兜底，防止连接泄漏 |
| 3 | 临时切换模型（降级/灰度） | 主模型限流时切备用模型 | 恢复逻辑放 `finally`，区分「原本不存在」的变量 |
| 4 | 批量 prompt 文件处理 | 文件数量运行时才知道 | `ExitStack` 是唯一干净方案 |
| 5 | 全链路计时（装饰器+with 两用） | 给任意 LLM 调用函数挂指标 | 用 `ContextDecorator`，一处实现两处可用 |

In [32]:
# 模式 1：LLM 客户端会话 —— 自动统计调用量，退出时释放并输出计费信息
class LLMClient:
    """模拟 LLM API 客户端：会话级用量统计 + 自动释放"""
    def __init__(self, model):
        self.model = model
        self.calls = 0
        self.tokens = 0
    def __enter__(self):
        print(f"[连接] 建立到 {self.model} 的会话")
        return self
    def chat(self, prompt):
        self.calls += 1
        self.tokens += len(prompt) * 2          # 模拟 token 计费
        return f"[{self.model}] 回复: {prompt[:10]}..."
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"[关闭] 会话调用 {self.calls} 次，消耗 {self.tokens} tokens（计费依据）")
        return False

with LLMClient("qwen-plus") as client:
    client.chat("总结上下文管理器的核心价值")
    client.chat("写一个 Python 示例")
# 适用：任何"会话级统计 + 必须释放"的资源（DB 连接、HTTP Session、GPU 句柄）

[连接] 建立到 qwen-plus 的会话
[关闭] 会话调用 2 次，消耗 52 tokens（计费依据）


In [33]:
# 模式 2：流式响应读取 —— closing 保证连接必然释放
from contextlib import closing

def fake_sse_stream():
    """模拟 LLM 流式接口：响应对象只有 close()，没有实现 CM 协议"""
    class FakeResponse:
        def __init__(self):
            self.chunks = iter(["上下文", "管理器", "保证", "清理", "..."])
            self.closed = False
        def read_chunk(self):
            return next(self.chunks, "")
        def close(self):
            self.closed = True
    return FakeResponse()

with closing(fake_sse_stream()) as resp:
    text = ""
    while chunk := resp.read_chunk():
        text += chunk
    print("拼接结果:", text)
print("响应对象已关闭:", resp.closed)
# 注意：若忘记 closing，一旦解析中途抛异常，底层 TCP 连接就泄漏了

拼接结果: 上下文管理器保证清理...
响应对象已关闭: True


In [34]:
# 模式 3：临时切换模型 —— 主模型限流时切备用模型，用完自动恢复
import os
from contextlib import contextmanager

@contextmanager
def use_model(model):
    old = os.environ.get("LLM_MODEL")
    os.environ["LLM_MODEL"] = model
    try:
        yield model
    finally:
        if old is None:
            os.environ.pop("LLM_MODEL", None)
        else:
            os.environ["LLM_MODEL"] = old

os.environ["LLM_MODEL"] = "qwen-max"           # 线上主模型

def call_llm(prompt):
    return f"[{os.environ['LLM_MODEL']}] {prompt}"

print("正常流量 :", call_llm("你好"))
with use_model("qwen-turbo"):                  # 降级窗口
    print("降级窗口 :", call_llm("你好"))
print("自动恢复 :", call_llm("你好"))
# 注意：恢复逻辑必须在 finally；变量原本不存在时要 pop 而不是留脏值

正常流量 : [qwen-max] 你好
降级窗口 : [qwen-turbo] 你好
自动恢复 : [qwen-max] 你好


In [35]:
# 模式 4：批量 prompt 文件处理 —— ExitStack 管理数量不定的文件
from contextlib import ExitStack
import tempfile, os

tmpdir = tempfile.mkdtemp()
paths = []
for i in range(3):
    p = os.path.join(tmpdir, f"case_{i}.prompt")
    with open(p, "w", encoding="utf-8") as f:
        f.write(f"测试用例 {i}：请解释 RAG")
    paths.append(p)

def load_prompts(paths):
    """文件数量 N 运行时才确定 —— 无法写成一行 with，ExitStack 登场"""
    with ExitStack() as stack:
        files = [stack.enter_context(open(p, encoding="utf-8")) for p in paths]
        return [f.read() for f in files]      # return 触发 stack 逆序关闭所有文件

for item in load_prompts(paths):
    print("已加载:", item)

for p in paths:
    os.remove(p)
os.rmdir(tmpdir)

已加载: 测试用例 0：请解释 RAG
已加载: 测试用例 1：请解释 RAG
已加载: 测试用例 2：请解释 RAG


In [36]:
# 模式 5：全链路计时 —— ContextDecorator 一处实现，装饰器/with 两用
from contextlib import ContextDecorator
import time

class llm_timer(ContextDecorator):
    def __init__(self, label="LLM 调用"):
        self.label = label
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    def __exit__(self, *exc):
        print(f"[metric] {self.label} 耗时 {time.perf_counter() - self.start:.4f}s")
        return False

@llm_timer("生成摘要")          # 用法 A：装饰器，整个函数计时
def summarize(text):
    time.sleep(0.02)
    return "摘要: " + text[:6]

print(summarize("上下文管理器是 Python 中..."))

with llm_timer("批量推理"):     # 用法 B：with 块，任意代码段计时
    time.sleep(0.01)
print("完成")

[metric] 生成摘要 耗时 0.0247s
摘要: 上下文管理器
[metric] 批量推理 耗时 0.0102s
完成


## 12. 内置 / 生态能力对照表

> **本章解决什么问题**：遇到一个资源，该用哪个现成能力？一张表查完。

| 资源 / 需求 | 标准库 / 生态能力 | 退出时行为 | 备注 |
| --- | --- | --- | --- |
| 文件 | `open()` 返回的文件对象 | 关闭文件 | 内置 CM 协议 |
| 线程/进程锁 | `threading.Lock`、`multiprocessing.Lock` | 释放锁 | 死锁风险见第 14 章 |
| SQLite 事务 | `with sqlite3.Connection` | **提交/回滚事务（不关闭连接）** | 高频误区！ |
| 关闭任意对象 | `contextlib.closing` | 调用 `.close()` | 只要求有 close 方法 |
| 抑制异常 | `contextlib.suppress` | 按名单吞异常 | 替代 try/except/pass |
| 重定向输出 | `contextlib.redirect_stdout/stderr` | 恢复原输出 | 日志归集常用 |
| 动态多资源 | `contextlib.ExitStack` | 逆序清理全部 | 数量不定唯一解 |
| 占位 CM | `contextlib.nullcontext` | 无操作 | 统一「可选」代码路径 |
| CM+装饰器两用 | `contextlib.ContextDecorator` | 取决于实现 | 计时/日志利器 |
| 临时目录 | `tempfile.TemporaryDirectory` | 递归删除目录 | 测试/缓存首选 |
| 测试替换 | `unittest.mock.patch` | 还原被替换对象 | 测试隔离基石 |
| HTTP 会话 | `requests.Session`、`httpx.Client` | 关闭连接池 | 生产环境务必用 with |
| 异步 HTTP/LLM | `aiohttp.ClientSession`、`openai.AsyncOpenAI` | 异步关闭 | 必须 `async with` |
| DB ORM 会话 | `sqlalchemy.Session` | 关闭会话 | 配合事务 CM 使用 |

## 13. 综合实战：LLMOps 批量推理任务

> **本章解决什么问题**：把前面 6 个知识点组合成一个贴近真实业务的批量推理管道。

**业务背景**：对一批 prompt 批量调用 LLM，要求：

1. 全程写运行日志到文件（资源必须关闭）—— `ExitStack`；
2. 主模型第 1 次调用必然限流（确定性模拟），限流后**降级到备用模型**重试 —— `suppress` + 环境变量切换 CM；
3. 每条与整体都计时，汇总成指标 —— 生成器版计时 CM；
4. 任何一条失败不中断整批 —— 精确抑制而非全局吞异常。

**设计取舍**（先想清楚再看代码）：

| 决策点 | 选择 | 理由 |
| --- | --- | --- |
| 类 vs 生成器 | 计时、环境切换用 `@contextmanager` | 逻辑是单向的「准备-清理」，无复杂状态 |
| 日志文件管理 | `ExitStack.enter_context` 而非直接 `with open` | 与后续可能追加的资源（如结果文件）统一逆序管理 |
| 限流处理 | `suppress(LLMRateLimitError)` 后判空降级 | 只吞明确的业务异常；`result is None` 作为降级信号 |
| 降级实现 | 环境变量切换 CM 包裹备用 client | 生产环境中模型名通常走配置中心/环境变量 |

In [37]:
# 实战 Part 1：基础设施（模拟 LLM 平台 + 三个工具 CM）
import os
import time
from contextlib import contextmanager, suppress

class LLMRateLimitError(Exception):
    """模拟 LLM 平台限流异常"""

class FakeLLMClient:
    """确定性模拟：前 fail_times 次调用抛限流，之后恢复正常"""
    def __init__(self, model, fail_times=0):
        self.model = model
        self.fail_times = fail_times
        self.calls = 0
    def chat(self, prompt):
        self.calls += 1
        if self.calls <= self.fail_times:
            raise LLMRateLimitError(f"{self.model} 限流（第 {self.calls} 次调用）")
        return f"[{self.model}] 回复「{prompt[:8]}」"

@contextmanager
def llm_env(**kwargs):
    """临时修改环境变量，退出恢复原状（模式 3 的通用化）"""
    old = {k: os.environ.get(k) for k in kwargs}
    for k, v in kwargs.items():
        os.environ[k] = str(v)
    try:
        yield
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v

@contextmanager
def stage_timer(label, sink):
    """计时并把结果追加到指标列表（生成器版 CM）"""
    start = time.perf_counter()
    try:
        yield
    finally:
        sink.append(f"{label}: {time.perf_counter() - start:.4f}s")

print("基础设施就绪：FakeLLMClient / llm_env / stage_timer")

基础设施就绪：FakeLLMClient / llm_env / stage_timer


In [38]:
# 实战 Part 2：批量推理管道 —— 组合 ExitStack + suppress + llm_env + stage_timer
import os
import tempfile
from contextlib import ExitStack, suppress

def run_batch(prompts, log_path):
    client = FakeLLMClient("qwen-max", fail_times=1)   # 第 1 次调用必然限流（确定性）
    metrics = []
    results = []

    with ExitStack() as stack, stage_timer("整批总耗时", metrics):
        log = stack.enter_context(open(log_path, "a", encoding="utf-8"))

        for i, prompt in enumerate(prompts):
            with stage_timer(f"第{i + 1}条", metrics):
                result = None
                with suppress(LLMRateLimitError):      # 只吞限流这一种业务异常
                    result = client.chat(prompt)
                if result is None:                     # 被限流 -> 降级到备用模型
                    print(f"  [降级] 第{i + 1}条主模型限流，切换备用模型重试")
                    with llm_env(LLM_MODEL="qwen-turbo"):
                        result = FakeLLMClient(os.environ["LLM_MODEL"]).chat(prompt)
                results.append(result)
                log.write(f"{prompt} -> {result}\n")

    print("推理结果:")
    for r in results:
        print(" ", r)
    print("性能指标:")
    for m in metrics:
        print(" ", m)
    print("日志文件已关闭:", log.closed)

tmpdir = tempfile.mkdtemp()
log_path = os.path.join(tmpdir, "batch.log")
run_batch(["解释上下文管理器", "写一个装饰器例子"], log_path)

with open(log_path, encoding="utf-8") as f:
    print("日志文件内容:", f.read().strip().splitlines())
os.remove(log_path)
os.rmdir(tmpdir)

  [降级] 第1条主模型限流，切换备用模型重试


推理结果:
  [qwen-turbo] 回复「解释上下文管理器」
  [qwen-max] 回复「写一个装饰器例子」
性能指标:
  第1条: 0.0001s
  第2条: 0.0000s
  整批总耗时: 0.0050s
日志文件已关闭: True


日志文件内容: ['解释上下文管理器 -> [qwen-turbo] 回复「解释上下文管理器」', '写一个装饰器例子 -> [qwen-max] 回复「写一个装饰器例子」']


**复盘**：这个 30 行的管道用到了——

1. `ExitStack`：日志文件随用随注册，后续加结果文件零改动；
2. `suppress`：精确吞限流，其他异常（如参数错误）照样暴露；
3. `llm_env`：降级窗口结束自动还原模型配置，不污染后续请求；
4. `stage_timer`：每条/整批两级计时，指标统一进 `metrics` 列表；
5. 全部清理逻辑都在 `finally`/`__exit__` 中，中途任何异常都不会泄漏文件句柄。

## 14. 常见陷阱汇总

> **本章解决什么问题**：一张总表 + 两个最容易踩的陷阱的可运行复现（其余陷阱已在第 10 章反例单元逐一复现）。

| # | 陷阱 | 症状 | 解法 |
| --- | --- | --- | --- |
| 1 | `__exit__` 无条件返回 True | bug 静默消失 | 只抑制明确异常（§10.1） |
| 2 | 以为 `as` 变量是管理器本身 | 类型不符、属性报错 | 记住 `as` = `__enter__` 返回值（§10.2） |
| 3 | 生成器 yield 次数 ≠ 1 | `RuntimeError` | 恰好一次（§10.3） |
| 4 | 清理代码再抛异常 | 原始异常被覆盖 | 清理逻辑 try/except 保护（§10.4） |
| 5 | 同步/异步 CM 混用 | `TypeError` | 协议与语句对应（§10.5） |
| 6 | 忘记多资源是**后进先出** | 依赖顺序错误（如先关日志再关业务） | 下面复现 1 |
| 7 | 以为 `with sqlite3.Connection` 会关连接 | 连接泄漏 | 它只管事务；关连接用 `closing`，下面复现 2 |
| 8 | 清理逻辑没放 `finally` | 异常路径资源泄漏 | `@contextmanager` 中 yield 后代码包 finally（§4） |

In [39]:
# 陷阱 6 复现：多资源释放是后进先出 —— 顺序错了日志就丢
class NamedResource:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print(f"打开 {self.name}")
        return self
    def __exit__(self, *exc):
        print(f"关闭 {self.name}")
        return False

print("进入顺序: logger -> db；退出顺序必然是 db -> logger（栈）")
with NamedResource("logger"), NamedResource("db"):
    print("  工作中")
print("结论：如果业务资源依赖 logger，应让 logger 先进入、后退出 —— 恰是默认行为，但要心里有数")

进入顺序: logger -> db；退出顺序必然是 db -> logger（栈）
打开 logger
打开 db
  工作中
关闭 db
关闭 logger
结论：如果业务资源依赖 logger，应让 logger 先进入、后退出 —— 恰是默认行为，但要心里有数


In [40]:
# 陷阱 7 复现：with sqlite3.Connection 只管理事务，不关闭连接
import sqlite3

conn = sqlite3.connect(":memory:")
with conn:                                   # 提交事务
    conn.execute("CREATE TABLE t (x INTEGER)")

try:
    with conn:                               # 中途异常 -> 回滚
        conn.execute("INSERT INTO t VALUES (1)")
        raise RuntimeError("中途失败")
except RuntimeError:
    pass

print("回滚后表内行数:", conn.execute("SELECT COUNT(*) FROM t").fetchone()[0])
print("with 结束后连接仍可用（未关闭）:", conn.execute("SELECT 1").fetchone())
conn.close()                                 # 连接要手动关，或用 closing()
print("结论：sqlite3 的 with = 事务管理器；关连接是另一回事")

回滚后表内行数: 0
with 结束后连接仍可用（未关闭）: (1,)
结论：sqlite3 的 with = 事务管理器；关连接是另一回事


## 15. 总结

### 15.1 记忆口诀

1. **一进一出必然清理**：`__enter__` 准备，`__exit__` 清理，异常也跑不掉；
2. **as 拿的是返回值**：`as` 变量 = `__enter__` 的 return，不一定是管理器本身；
3. **True 吞异常要克制**：`__exit__` 返回真值 = 吞掉异常，只吞你点名的；
4. **yield 恰好一次**：`@contextmanager` 的生成器，`yield` 前后分别是进和出，清理放 `finally`；
5. **多资源后进先出**：进入左到右、退出右到左，`ExitStack` 管数量不定的资源；
6. **同步异步不混用**：`__enter__` 配 `with`，`__aenter__` 配 `async with`。

In [41]:
# 15.2 通用代码模板（可直接复制改造）
# ---------- 模板 A：类实现（有状态、需复用、逻辑复杂时选它） ----------
class MyResource:
    def __init__(self, *args, **kwargs):
        pass                     # ① 构造时只存参数，真正获取资源放 __enter__

    def __enter__(self):
        # ② 获取资源（打开连接/文件/锁），返回 as 变量需要的对象
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # ③ 释放资源；清理逻辑要健壮（内部 try/except 保护）
        # ④ return True 抑制异常；return False/None 正常传播（推荐默认 False）
        return False

with MyResource() as r:
    print("模板 A 运行中:", type(r).__name__)

# ---------- 模板 B：@contextmanager（简单准备/清理时选它，更短） ----------
from contextlib import contextmanager

@contextmanager
def my_resource():
    resource = "资源"            # ① yield 前：进入逻辑（获取资源）
    try:
        yield resource           # ② 恰好 yield 一次；值 -> as 变量
    finally:
        pass                     # ③ yield 后：清理逻辑，必须放 finally

with my_resource() as r:
    print("模板 B 运行中:", r)

模板 A 运行中: MyResource
模板 B 运行中: 资源


### 15.3 自测练习题

> 先自己实现，再对照下面的参考实现（参考实现已实际执行）。

1. **练习 1（类实现）**：实现 `Tag(name)`，`with` 嵌套时打印成对的开闭标签，如 `<prompt>` … `</prompt>`；
2. **练习 2（生成器）**：实现 `timeit_block(label)`，要求**即使块内抛异常也打印耗时**，且异常继续传播；
3. **练习 3（综合）**：给定一组资源名，用 `ExitStack + closing` 统一打开并保证逆序关闭。

In [42]:
# 练习 1 参考实现：成对打印标签
class Tag:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print(f"<{self.name}>")
        return self
    def __exit__(self, *exc):
        print(f"</{self.name}>")
        return False

with Tag("prompt"):
    with Tag("system"):
        print("  你是一个严谨的助手")
    with Tag("user"):
        print("  解释上下文管理器")

<prompt>
<system>
  你是一个严谨的助手
</system>
<user>
  解释上下文管理器
</user>
</prompt>


In [43]:
# 练习 2 参考实现：异常也要打印耗时
from contextlib import contextmanager
import time

@contextmanager
def timeit_block(label):
    start = time.perf_counter()
    try:
        yield
    finally:                                  # finally -> 异常路径也打印
        print(f"{label} 耗时 {time.perf_counter() - start:.4f}s")

with timeit_block("正常任务"):
    sum(range(100_000))

try:
    with timeit_block("会失败的任务"):
        raise ValueError("boom")
except ValueError as e:
    print("异常正常传播:", e)

正常任务 耗时 0.0007s
会失败的任务 耗时 0.0000s
异常正常传播: boom


In [44]:
# 练习 3 参考实现：ExitStack + closing 统一管理资源
from contextlib import ExitStack, closing

class FakeConn:
    def __init__(self, name):
        self.name = name
    def close(self):
        print(f"关闭 {self.name}")

def run_pipeline(names):
    with ExitStack() as stack:
        conns = [stack.enter_context(closing(FakeConn(n))) for n in names]
        print("使用资源:", [c.name for c in conns])

run_pipeline(["db", "cache", "llm"])
# 观察输出：关闭顺序与打开顺序相反（后进先出）

使用资源: ['db', 'cache', 'llm']
关闭 llm
关闭 cache
关闭 db


---

✅ **全篇完**。建议动手路径：先独立做完 3 道练习 → 把第 13 章实战改成对接你真实的 LLM SDK → 回头重读第 2 章协议与修正说明。